In [1]:
from data_utils import HarmonicGraphDataset, harmonic_graph_collate_fn
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
import torch
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss

from models_graph import HarmonicGraphEncoder
from generate_utils import load_AttnFiLMSEModel

import pickle

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the dataset and the tokenizer
with open("data/gjt_train.pkl", "rb") as f:
    gjt_train = pickle.load(f)

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

graph_model = HarmonicGraphEncoder()
graph_model.to(device)

# load the model
model = load_AttnFiLMSEModel(
    tokenizer=tokenizer,
    guidance_dim=graph_model.output_dim,
    device=device
)
model.freeze_base()

In [ ]:
dataset = HarmonicGraphDataset(gjt_train, tokenizer, model)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=harmonic_graph_collate_fn)

In [5]:
# d = dataset[246]
# print(d)

In [ ]:
batch = next(iter(dataloader))
print(batch.keys())

dict_keys(['pianoroll', 'real_harmony_ids', 'recomposed_harmony_ids', 'random_harmony_ids', 'real_graph', 'recomposed_graph', 'random_graph', 'mask_token_positions'])


In [40]:
print(batch['pianoroll'].shape)
guide_z = graph_model(batch['real_graph'].to(model.device))
print(guide_z.shape)
constraints = batch['real_harmony_ids'].clone()
constraints[batch['mask_token_positions']] = tokenizer.mask_token_id
print(constraints.shape)
print(constraints.dtype)

torch.Size([16, 80, 13])
torch.Size([16, 128])
torch.Size([16, 80])
torch.int64


In [41]:
real_guide_z = graph_model(batch['real_graph'].to(model.device))
real_constraints = batch['real_harmony_ids'].clone()
real_constraints[batch['mask_token_positions']] = tokenizer.mask_token_id

recomposed_guide_z = graph_model(batch['recomposed_graph'].to(model.device))
recomposed_constraints = batch['recomposed_harmony_ids'].clone()
recomposed_constraints[batch['mask_token_positions']] = tokenizer.mask_token_id

random_guide_z = graph_model(batch['random_graph'].to(model.device))
random_constraints = batch['real_harmony_ids'].clone()
random_constraints[batch['mask_token_positions']] = tokenizer.mask_token_id

In [42]:
real_logits = model(
    batch['pianoroll'].to(model.device),
    real_constraints.to(model.device),
    real_guide_z.to(model.device)
)

recomposed_logits = model(
    batch['pianoroll'].to(model.device),
    recomposed_constraints.to(model.device),
    recomposed_guide_z.to(model.device)
)

random_logits = model(
    batch['pianoroll'].to(model.device),
    random_constraints.to(model.device),
    random_guide_z.to(model.device)
)

In [43]:
logits_loss_fn = CrossEntropyLoss(ignore_index=-100)

In [44]:
real_logits_loss = logits_loss_fn(real_logits.view(-1, real_logits.size(-1)), batch['real_harmony_ids'].view(-1))
recomposed_logits_loss = logits_loss_fn(recomposed_logits.view(-1, recomposed_logits.size(-1)), batch['recomposed_harmony_ids'].view(-1))
random_logits_loss = logits_loss_fn(random_logits.view(-1, random_logits.size(-1)), batch['random_harmony_ids'].view(-1))

In [45]:
print(real_logits_loss)
print(recomposed_logits_loss)
print(random_logits_loss)

tensor(0.4207, grad_fn=<NllLossBackward0>)
tensor(0.7187, grad_fn=<NllLossBackward0>)
tensor(3.1555, grad_fn=<NllLossBackward0>)
